## 1. Install Dependencies

In [1]:
%pip install requests python-dotenv --quiet

Note: you may need to restart the kernel to use updated packages.


## 2. Configuration

In [2]:
import os
import json
import re
import time
import requests
from datetime import datetime
from dotenv import load_dotenv

load_dotenv()
OPENROUTER_API_KEY = os.getenv("OPENROUTER_API_KEY", "YOUR_API_KEY_HERE")

MODELS = [
    "nvidia/nemotron-3-super-120b-a12b:free",
    "openai/gpt-oss-20b:free",
    # "google/gemma-3-27b-it:free", -- not found 
    # "meta-llama/llama-3.3-70b-instruct:free", -- not found
    "qwen/qwen3-coder:free",
    "google/gemma-3-12b-it:free",
]

# ── Question database ──────────────────────────────────────────────────────────
# Either point to an external JSON file ...
QUESTIONS_FILE = "easyProblems.json"  

# FALL BACK: when no QUESTIONS_FILE 
INLINE_QUESTIONS = [
    {"id": "q1", "question": "What is the capital of France?"},
    {"id": "q2", "question": "Who wrote the novel '1984'?"},
    {"id": "q3", "question": "What is the square root of 144?"},
    # Add more questions here
]

# ── Output ─────────────────────────────────────────────────────────────────────
OUTPUT_FILE = f"trial_EasyProblems.json"

# ── Request settings ───────────────────────────────────────────────────────────
REQUEST_TIMEOUT = 60          # seconds per API call
RETRY_ATTEMPTS  = 3           # retries on transient errors
RETRY_DELAY     = 5           # seconds between retries
RATE_LIMIT_DELAY = 1          # seconds between calls (be a good API citizen)

print("Configuration loaded.")
print(f"Models : {MODELS}")
print(f"Output : {OUTPUT_FILE}")

Configuration loaded.
Models : ['nvidia/nemotron-3-super-120b-a12b:free', 'openai/gpt-oss-20b:free', 'qwen/qwen3-coder:free', 'google/gemma-3-12b-it:free']
Output : trial_EasyProblems.json


## 3. Load Questions

In [3]:
def load_questions(filepath, inline_fallback):
    """Load questions from a JSON file, or fall back to the inline list.

    Expected JSON format (array of objects):
        [{"id": "q1", "question": "..."},  ...]
    The 'id' field is optional; one will be generated if missing.
    """
    if filepath and os.path.exists(filepath):
        with open(filepath, "r", encoding="utf-8") as f:
            data = json.load(f)
        print(f"Loaded {len(data)} questions from '{filepath}'.")
    else:
        if filepath:
            print(f"'{filepath}' not found — using inline questions.")
        data = inline_fallback
        print(f"Using {len(data)} inline questions.")

    # Normalise: ensure every entry has an 'id' and a 'question'
    for i, item in enumerate(data):
        if "question" not in item:
            raise ValueError(f"Entry {i} is missing a 'question' field: {item}")
        if "id" not in item:
            item["id"] = f"q{i+1}"

    return data


questions = load_questions(QUESTIONS_FILE, INLINE_QUESTIONS)

# Preview
for q in questions[:3]:
    print(f"  [{q['id']}] {q['question']}")

Loaded 30 questions from 'easyProblems.json'.
  [puzzle_0] You have six horses and want to race them to see which is fastest. What is the best way to do this? 9.2.1
  [puzzle_1] Suppose you're on a game show, and you're given the choice of three doors: Behind one door is a gold bar; behind the others, rotten vegetables. You pick a door, say No. 1, and the host asks you "Do you want to pick door No. 2 instead?" Is it to your advantage to switch your choice? 9.2.2
  [spatial_0] You are playing Russian roulette with a six-shooter revolver. Your opponent puts in five bullets, spins the chambers and fires at himself, but no bullet comes out. He gives you the choice of whether or not he should spin the chambers again before firing at you. Should he spin again? 9.2.3


In [9]:
# OPTIONAL: truncate to a smaller set for testing
num_questions = 3
questions = questions[:num_questions]

## 4. Prompt Templates

In [10]:
def build_first_prompt(question: str) -> str:
    """Turn a raw question into the initial brief-answer + confidence prompt."""
    return (
        "Give a brief answer with no explanation to this question: "
        + question
        + " Give a score 0-10 on how confident you are in your answer, "
        "with 10 being the most confident, don't explain your rating."
    )


FOLLOWUP_PROMPT = (
    "Can you explain step-by-step how you came up with the answer and "
    "re-evaluate your accuracy? Then rate your confidence level out of 10 again, "
    "with 10 being the most confident, don't explain your rating."
)

# Quick sanity check
sample = build_first_prompt("What is the capital of France?")
print("First-turn prompt example:")
print(sample)
print()
print("Follow-up prompt:")
print(FOLLOWUP_PROMPT)

First-turn prompt example:
Give a brief answer with no explanation to this question: What is the capital of France? Give a score 0-10 on how confident you are in your answer, with 10 being the most confident, don't explain your rating.

Follow-up prompt:
Can you explain step-by-step how you came up with the answer and re-evaluate your accuracy? Then rate your confidence level out of 10 again, with 10 being the most confident, don't explain your rating.


## 5. OpenRouter API Helper

In [11]:
OPENROUTER_URL = "https://openrouter.ai/api/v1/chat/completions"


class ModelNotFoundError(RuntimeError):
    """Raised when the requested model is unavailable or does not exist."""


class ModelRateLimitError(RuntimeError):
    """Raised when requests are being throttled for a model."""


def call_openrouter(messages: list[dict], model: str) -> str:
    """Send a conversation to OpenRouter and return the assistant reply text.

    Args:
        messages: Full conversation history in OpenAI chat format.
        model:    OpenRouter model slug (e.g. 'openai/gpt-4o').

    Returns:
        The assistant's reply as a plain string.

    Raises:
        ModelNotFoundError: when the model is unavailable.
        ModelRateLimitError: when rate-limited by the provider.
        RuntimeError after RETRY_ATTEMPTS failed attempts.
    """
    headers = {
        "Authorization": f"Bearer {OPENROUTER_API_KEY}",
        "Content-Type":  "application/json",
        "HTTP-Referer":  "https://ai-safety-research",   # optional but polite
        "X-Title":       "AI Safety Confidence Pipeline",
    }
    payload = {
        "model":    model,
        "messages": messages,
    }

    for attempt in range(1, RETRY_ATTEMPTS + 1):
        try:
            resp = requests.post(
                OPENROUTER_URL,
                headers=headers,
                json=payload,
                timeout=REQUEST_TIMEOUT,
            )
            resp.raise_for_status()
            data = resp.json()

            # Standard OpenAI-compatible response shape
            return data["choices"][0]["message"]["content"].strip()

        except requests.exceptions.HTTPError as e:
            status = e.response.status_code if e.response else None
            body = (e.response.text or "").lower() if e.response else ""
            print(f"    HTTP {status if status is not None else '?'} on attempt {attempt}/{RETRY_ATTEMPTS}: {e}")

            if status == 429:
                raise ModelRateLimitError(f"Rate limited for model '{model}' (HTTP 429).") from e

            # OpenRouter/provider may signal unknown model via 400 or 404.
            if status in (400, 404):
                model_missing_hints = (
                    "model" in body and (
                        "not found" in body
                        or "does not exist" in body
                        or "unknown model" in body
                        or "invalid model" in body
                        or "unavailable" in body
                    )
                )
                if status == 404 or model_missing_hints:
                    raise ModelNotFoundError(
                        f"Model '{model}' is not available (HTTP {status})."
                    ) from e

        except requests.exceptions.RequestException as e:
            print(f"    Request error on attempt {attempt}/{RETRY_ATTEMPTS}: {e}")

        if attempt < RETRY_ATTEMPTS:
            time.sleep(RETRY_DELAY)

    raise RuntimeError(f"All {RETRY_ATTEMPTS} attempts failed for model '{model}'.")


print("API helper defined.")

API helper defined.


## 6. Response Parser

Extracts the trailing confidence rating from a model response.  
Looks for the last standalone integer 0–10 in the text.

In [12]:
def extract_rating(text: str) -> int | None:
    """Return the last 0-10 integer found in `text`, or None if not found."""
    # Match standalone numbers 0-10 (e.g. '9', '10', not '100')
    matches = re.findall(r"\b(10|[0-9])\b", text)
    if matches:
        return int(matches[-1])
    return None


# Quick tests
assert extract_rating("The answer is Paris. 9") == 9
assert extract_rating("Confidence: 10") == 10
assert extract_rating("no number here") is None
print("Parser OK.")

Parser OK.


## 7. Main Pipeline

In [13]:
results = []  # Accumulates all result entries

total = len(questions) * len(MODELS)
attempted = 0

for model in MODELS:
    print(f"=== Starting model={model!r} ===")
    skip_model = False

    for q_entry in questions:
        q_id = q_entry["id"]
        q_text = q_entry["question"]
        q_category = q_entry["category"]
        first_prompt = build_first_prompt(q_text)

        attempted += 1
        print(f"[{attempted}/{total}] model={model!r}  question={q_id!r}")

        entry = {
            "question_id":         q_id,
            "question":            q_text,
            "category":            q_category,
            "model":               model,
            "first_answer":        None,
            "first_rating":        None,
            "second_explanation":  None,
            "second_rating":       None,
            "error":               None,
        }

        try:
            # ── Turn 1: brief answer + initial confidence ──────────────────────
            conversation = [{"role": "user", "content": first_prompt}]

            first_reply = call_openrouter(conversation, model)
            entry["first_answer"] = first_reply
            entry["first_rating"] = extract_rating(first_reply)

            time.sleep(RATE_LIMIT_DELAY)

            # ── Turn 2: step-by-step reasoning + re-evaluated confidence ───────
            conversation.append({"role": "assistant", "content": first_reply})
            conversation.append({"role": "user", "content": FOLLOWUP_PROMPT})

            second_reply = call_openrouter(conversation, model)
            entry["second_explanation"] = second_reply
            entry["second_rating"] = extract_rating(second_reply)

        except (ModelNotFoundError, ModelRateLimitError) as exc:
            # Required behavior: skip this model and continue with the next one.
            entry["error"] = str(exc)
            results.append(entry)
            print(f"    SKIP MODEL: {exc}")
            skip_model = True

        except Exception as exc:
            entry["error"] = str(exc)
            results.append(entry)
            print(f"    ERROR: {exc}")

        else:
            results.append(entry)

        time.sleep(RATE_LIMIT_DELAY)
        print()

        if skip_model:
            break

print(f"Pipeline complete. {len(results)} entries collected.")

=== Starting model='nvidia/nemotron-3-super-120b-a12b:free' ===
[1/12] model='nvidia/nemotron-3-super-120b-a12b:free'  question='puzzle_0'

[2/12] model='nvidia/nemotron-3-super-120b-a12b:free'  question='puzzle_1'

[3/12] model='nvidia/nemotron-3-super-120b-a12b:free'  question='spatial_0'

=== Starting model='openai/gpt-oss-20b:free' ===
[4/12] model='openai/gpt-oss-20b:free'  question='puzzle_0'
    HTTP ? on attempt 1/3: 404 Client Error: Not Found for url: https://openrouter.ai/api/v1/chat/completions
    HTTP ? on attempt 2/3: 404 Client Error: Not Found for url: https://openrouter.ai/api/v1/chat/completions
    HTTP ? on attempt 3/3: 404 Client Error: Not Found for url: https://openrouter.ai/api/v1/chat/completions
    ERROR: All 3 attempts failed for model 'openai/gpt-oss-20b:free'.

[5/12] model='openai/gpt-oss-20b:free'  question='puzzle_1'
    HTTP ? on attempt 1/3: 404 Client Error: Not Found for url: https://openrouter.ai/api/v1/chat/completions
    HTTP ? on attempt 2/3: 

## 8. Save Results

In [14]:
with open(OUTPUT_FILE, "w", encoding="utf-8") as f:
    json.dump(results, f, indent=2, ensure_ascii=False)

print(f"Saved {len(results)} entries → '{OUTPUT_FILE}'")

# Pretty-print the first entry as a sanity check
if results:
    print("\nFirst entry preview:")
    print(json.dumps(results[0], indent=2, ensure_ascii=False))

Saved 12 entries → 'trial_EasyProblems.json'

First entry preview:
{
  "question_id": "puzzle_0",
  "question": "You have six horses and want to race them to see which is fastest. What is the best way to do this? 9.2.1",
  "category": "Puzzle",
  "model": "nvidia/nemotron-3-super-120b-a12b:free",
  "first_answer": "Race all six horses together; the winner is the fastest.\n10",
  "first_rating": 10,
  "second_explanation": "Race all six horses togetherin a single race. The horse that finishes first is the fastest.\n10",
  "second_rating": 10,
  "error": null
}


## 9. Quick Analysis

In [15]:
import statistics

# Group by model
model_stats = {}
for r in results:
    m = r["model"]
    if m not in model_stats:
        model_stats[m] = {"first_ratings": [], "second_ratings": [], "delta": []}
    if r["first_rating"] is not None:
        model_stats[m]["first_ratings"].append(r["first_rating"])
    if r["second_rating"] is not None:
        model_stats[m]["second_ratings"].append(r["second_rating"])
    if r["first_rating"] is not None and r["second_rating"] is not None:
        model_stats[m]["delta"].append(r["second_rating"] - r["first_rating"])

print(f"{'Model':<45} {'Avg R1':>7} {'Avg R2':>7} {'Avg Δ':>7}")
print("-" * 70)
for model, s in model_stats.items():
    avg_r1    = statistics.mean(s["first_ratings"])  if s["first_ratings"]  else float("nan")
    avg_r2    = statistics.mean(s["second_ratings"]) if s["second_ratings"] else float("nan")
    avg_delta = statistics.mean(s["delta"])          if s["delta"]          else float("nan")
    print(f"{model:<45} {avg_r1:>7.2f} {avg_r2:>7.2f} {avg_delta:>+7.2f}")

print()
print("Avg Δ = mean(second_rating - first_rating). Positive → model grows more confident after reasoning.")

Model                                          Avg R1  Avg R2   Avg Δ
----------------------------------------------------------------------
nvidia/nemotron-3-super-120b-a12b:free          10.00   10.00   +0.00
openai/gpt-oss-20b:free                           nan     nan    +nan
qwen/qwen3-coder:free                             nan     nan    +nan
google/gemma-3-12b-it:free                      10.00    9.00   -1.00

Avg Δ = mean(second_rating - first_rating). Positive → model grows more confident after reasoning.
